In [ ]:
import torch
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from omegaconf import OmegaConf

from utils._utils import *
from utils.test_utils import *
from models.registry import MODEL_REGISTRY

import models.models

save_path = "checkpoints/exp6(34.35)/20260403_211549"
save_path = "checkpoints/exp11/20260411_202723"
# save_path = "checkpoints/exp12/20260408_160145"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
config = OmegaConf.load(Path(save_path) / "config.yaml")
model = MODEL_REGISTRY.build(config.model.type, config)
best_model_path = Path(save_path) / "best.pth"
checkpoint = torch.load(best_model_path, map_location='cpu',  weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"], strict=False,)
s = config.data.scale
norm = config.data.get("norm", {'lr': {'sub': 0., 'div': 1.},'hr': {'sub': 0., 'div': 1.}})
model_name = config.model.type
data_root = config.data.data_path
model.to(device)
model.eval()
print(f"Model {model_name} loaded with {count_parameters(model):,} trainable parameters.")

Model CISFNO3 loaded with 118,106,952 trainable parameters.


In [ ]:
idx = 802
pic = Image.open(f"{data_root}/DIV2K_valid_LR_bicubic/X{s}/{idx:04d}x{s}.png").convert("RGB")
patch_h = 256
patch_w = 256
start_patch_h = 100
start_patch_w = 400
input_tensor = TF.to_tensor(pic).to(device)
patch_sheet_in = np.s_[:, start_patch_h:start_patch_h+patch_h, start_patch_w:start_patch_w+patch_w]
patch_sheet = np.s_[:, s*start_patch_h:s*start_patch_h+s*patch_h, s*start_patch_w:s*start_patch_w+s*patch_w]
if_patch_inference = True
with torch.no_grad():
    if if_patch_inference:
    #=========================patch inference=========================
        SR_tensor = tile_inference(model, input_tensor.unsqueeze(0), 
                                    patch_size=128,  # 和训练一致
                                    scale=s,
                                    only_head=False,
                                    norm=norm).squeeze(0).cpu()
        head_pred = tile_inference(model, input_tensor.unsqueeze(0), 
                                    patch_size=128,  # 和训练一致
                                    scale=s,
                                    only_head=True, 
                                    norm=norm).squeeze(0).cpu()
        SR_tensor = SR_tensor[patch_sheet]
        head_pred = head_pred[patch_sheet]
        res_pred = SR_tensor - head_pred
    else:
    #=========================sheet image inference=========================
        SR_tensor = direct_inference(model, input_tensor, patch_sheet_in, norm=norm).cpu()
        head_pred = direct_inference(model, input_tensor, patch_sheet_in, only_head=True, norm=norm).cpu()
        res_pred = SR_tensor - head_pred


gt_pic = Image.open(f"{data_root}/DIV2K_valid_HR/{idx:04d}.png").convert("RGB")
gt_tensor = TF.to_tensor(gt_pic).to('cpu')
gt_tensor = gt_tensor[patch_sheet]
head_res = gt_tensor - head_pred
res_tensor = gt_tensor - SR_tensor

_ , _ = visualize_results(gt_tensor, SR_tensor, idx=idx, s=s, title=f"SR vs GT for Image {idx:04d}", save_path = "./test_pic")
# _ , _ = visualize_results(gt_tensor, head_pred, idx=idx, s=s, title=f"head_pred vs GT for Image {idx:04d}", save_path = "./test_pic")


clim = (0.0,5.0)
channel_sheet = np.s_[0, :, :]
# visualize_distribution((gt_tensor-head_pred))
# visualize_distribution(flted_head_res)
# _,_ = fft_heatmap(gt_tensor[channel_sheet], title="GT Tensor FFT Magnitude Spectrum", log_scale=True,clim=None)
# _,_ = fft_heatmap(SR_tensor[channel_sheet], title="SR Tensor FFT Magnitude Spectrum", log_scale=True,clim=None)
# _,_ = fft_heatmap(res_tensor[channel_sheet], title="Res Tensor FFT Magnitude Spectrum", log_scale=True,clim=clim)
_,_ = fft_heatmap(res_pred[channel_sheet], title="Res pred FFT Magnitude Spectrum", log_scale=True,clim=clim)
_,_ = fft_heatmap(head_res[channel_sheet], title="Head_Res Tensor FFT Magnitude Spectrum", log_scale=True,clim=clim)

# _,_ = fft_heatmap((head_res[channel_sheet] / (res_pred[channel_sheet] + 1e-8)), title="Ratio Tensor FFT Magnitude Spectrum", log_scale=True,)

# data = (torch.log((head_res[channel_sheet] / (res_pred[channel_sheet] + 1e-8))) + 1e-8)

# # 清理非法值
# data = data[torch.isfinite(data)]

# # 再 clamp
# data = data.clamp(-100, 100)

# visualize_distribution(data)
# print(res_pred)

In [ ]:
def log_fft(arr):
    fft_result = torch.fft.fft2(arr)
    magnitude_spectrum = torch.log(torch.abs(fft_result) + 1)
    return magnitude_spectrum

L_F_head_res = log_fft(head_res[channel_sheet])
L_F_res_pred = log_fft(res_pred[channel_sheet])
ratio = L_F_res_pred / (L_F_head_res + 1e-8)

# normed_ratio = (ratio - ratio.min()) / (ratio.max() - ratio.min() + 1e-8)

print(ratio.max(), ratio.min(), ratio.mean(), ratio.std())

_,_ = fft_heatmap(ratio, title="Ratio Tensor FFT Magnitude Spectrum", log_scale=True,clim=None)
def remove_top_5_percent(data: torch.Tensor):
    """
    删除最大的5%数据，保留剩下95%
    Args:
        data: 1D tensor
    Returns:
        filtered_data: 过滤后的tensor
    """
    # 计算95%分位数
    threshold = torch.quantile(data, 0.99)
    
    # 过滤
    filtered_data = data[data <= threshold]
    
    return filtered_data

remove_top_ratio = remove_top_5_percent(ratio.flatten())

visualize_distribution(remove_top_ratio)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_derivatives(y, title="函数图像与各阶导数"):
    """
    绘制 y 及其一、二、三阶导数。

    参数:
        y     : array-like, shape=[N], 输入信号
        title : 图表标题
    """
    y = np.asarray(y, dtype=float)
    N = len(y)
    x = np.arange(N)

    # 用中心差分计算各阶导数（边界用前/后差分）
    def deriv(arr):
        d = np.empty_like(arr)
        d[1:-1] = (arr[2:] - arr[:-2]) / 2
        d[0]    = arr[1] - arr[0]
        d[-1]   = arr[-1] - arr[-2]
        return d

    d1 = deriv(y)
    d2 = deriv(d1)
    d3 = deriv(d2)

    curves = [
        (y,  "y  ", "#3478c8"),
        (d1, "y' ", "#d85a30"),
        (d2, "y''", "#1d9e75"),
        (d3, "y'''","#7f77dd"),
    ]

    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(title, fontsize=14)

    for ax, (data, label, color) in zip(axes, curves):
        ax.plot(x, data, color=color, linewidth=1.5)
        ax.set_ylabel(label, fontsize=10)
        ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
        ax.grid(True, alpha=0.2)

    axes[-1].set_xlabel("index")
    plt.tight_layout()
    plt.show()
_,y,_=fft_energy_analysis(gt_tensor[channel_sheet]-head_pred[channel_sheet], fig_title="Head_Res Energy Analysis",)

plot_derivatives(y, title="Head_Res Energy and its Derivatives")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))


def Norm(im):
    im.permute(1,2,0).numpy()
    return (im - im.min()).abs() / (im.max() - im.min() + 1e-8)

im0 = axs[0].imshow(head_pred.permute(1, 2, 0).numpy())
axs[0].set_title("Head Prediction")

im1 = axs[1].imshow(Norm(gt_tensor - head_pred).permute(1, 2, 0).numpy(),)
axs[1].set_title("gt_res_head")
# TF.to_pil_image((pred_res)).save("1.png")

im2 = axs[2].imshow((res_pred).permute(1, 2, 0).numpy())
axs[2].set_title("pred_res")

plt.tight_layout()
plt.show()
# print(pred_res.max(),(gt_tensor - pred_head).max())

In [ ]:
import random
from utils.metrics_utils import SRMetric
from utils._utils import PSNR, SSIM
psnr_y = SRMetric("psnr", True, 2)
psnr_rgb = SRMetric("psnr", False, 2)
ssim_y = SRMetric("ssim", True, 2)
ssim_rgb = SRMetric("ssim", False, 2)
psnr_o = PSNR()
ssim_o = SSIM().to(device)



def evaluate(model, device, scale, data_root, psnr_y, ssim_y, psnr_rgb, ssim_rgb):
    import gc
    psnr_y_value, ssim_y_value = 0.0, 0.0
    psnr_rgb_value, ssim_rgb_value = 0.0, 0.0
    IDX = list(range(801, 901))
    
    for idx in IDX:
        lr_pic = Image.open(f"{data_root}/DIV2K_valid_LR_bicubic/X{scale}/{idx:04d}x{scale}.png").convert("RGB")
        hr_pic = Image.open(f"{data_root}/DIV2K_valid_HR/{idx:04d}.png").convert("RGB")

        lr_tensor = TF.to_tensor(lr_pic).unsqueeze(0).to(device)
        hr_tensor = TF.to_tensor(hr_pic).to(device)
        with torch.no_grad():
            sr_tensor = tile_inference(model, lr_tensor,
                                patch_size=128,  # 和训练一致
                                scale=s,
                                norm = norm).squeeze(0)
            psnr_tem = psnr_y(sr_tensor, hr_tensor).item()
            ssim_tem = ssim_y(sr_tensor, hr_tensor).item()
            psnr_y_value += psnr_tem
            ssim_y_value += ssim_tem
            print(f"Image {idx:04d} - PSNR_y: {psnr_tem:.4f}, SSIM_y: {ssim_tem:.4f}")
            psnr_tem = psnr_rgb(sr_tensor, hr_tensor).item()
            ssim_tem = ssim_rgb(sr_tensor, hr_tensor).item()
            psnr_rgb_value += psnr_tem
            ssim_rgb_value += ssim_tem
            print(f"Image {idx:04d} - PSNR_rgb: {psnr_tem:.4f}, SSIM_rgb: {ssim_tem:.4f}")

        del lr_tensor, hr_tensor, sr_tensor
        torch.cuda.empty_cache()
        gc.collect()

    print(f"PSNR_y: {psnr_y_value / len(IDX):.4f}")
    print(f"SSIM_y: {ssim_y_value / len(IDX):.4f}")
    print(f"PSNR_rgb: {psnr_rgb_value / len(IDX):.4f}")
    print(f"SSIM_rgb: {ssim_rgb_value / len(IDX):.4f}")

# 调用
evaluate(model, device, scale=2, data_root=data_root, psnr_y=psnr_y, ssim_y=ssim_y, psnr_rgb=psnr_rgb, ssim_rgb=ssim_rgb)